## Bayesian Belief Network for Student Performance Prediction

### 1. Define the Network Structure (Nodes and Edges)

In [3]:
!pip install pgmpy

from pgmpy.models import DiscreteBayesianNetwork as BayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

# Define the Bayesian Network structure (DAG)
model = BayesianNetwork([
    ('StudyHours', 'Performance'),
    ('Attendance', 'Performance'),
    ('Sleep', 'Stress'),
    ('Stress', 'Performance')
])

print("Bayesian Network structure defined:")
print(model.nodes())
print(model.edges())

Bayesian Network structure defined:
['StudyHours', 'Performance', 'Attendance', 'Sleep', 'Stress']
[('StudyHours', 'Performance'), ('Attendance', 'Performance'), ('Sleep', 'Stress'), ('Stress', 'Performance')]


### 2. Assign Conditional Probability Tables (CPTs)

In [11]:
cpd_study_hours = TabularCPD(variable='StudyHours', variable_card=2,
                             values=[[0.6], [0.4]], # Low: 0.6, High: 0.4
                             state_names={'StudyHours': ['Low', 'High']})

cpd_attendance = TabularCPD(variable='Attendance', variable_card=2,
                            values=[[0.3], [0.7]], # Poor: 0.3, Good: 0.7
                            state_names={'Attendance': ['Poor', 'Good']})

cpd_sleep = TabularCPD(variable='Sleep', variable_card=2,
                       values=[[0.4], [0.6]], # Less: 0.4, Adequate: 0.6
                       state_names={'Sleep': ['Less', 'Adequate']})

cpd_stress = TabularCPD(variable='Stress', variable_card=2,
                        values=[[0.8, 0.3],
                                [0.2, 0.7]],
                        evidence=['Sleep'],
                        evidence_card=[2],
                        state_names={'Stress': ['High', 'Low'], 'Sleep': ['Less', 'Adequate']})

cpd_performance = TabularCPD(variable='Performance', variable_card=2,
                             values=[[0.1, 0.2, 0.3, 0.5, 0.4, 0.6, 0.7, 0.9],
                                     [0.9, 0.8, 0.7, 0.5, 0.6, 0.4, 0.3, 0.1]],
                             evidence=['StudyHours', 'Attendance', 'Stress'],
                             evidence_card=[2, 2, 2],
                             state_names={'Performance': ['Pass', 'Fail'],
                                          'StudyHours': ['Low', 'High'],
                                          'Attendance': ['Poor', 'Good'],
                                          'Stress': ['High', 'Low']})

model.add_cpds(cpd_study_hours, cpd_attendance, cpd_sleep, cpd_stress, cpd_performance)

print("\nModel consistency check:", model.check_model())


Model consistency check: True


### 3. Perform Inference using Variable Elimination

In [5]:
infer = VariableElimination(model)

print("\nInference engine initialized using Variable Elimination.")


Inference engine initialized using Variable Elimination.


### 4. Run Queries

#### Query 1: Prior probability of Performance (Pass/Fail)

In [6]:
query1_result = infer.query(variables=['Performance'])
print(query1_result)

+-------------------+--------------------+
| Performance       |   phi(Performance) |
+===================+====================+
| Performance(Pass) |             0.4790 |
+-------------------+--------------------+
| Performance(Fail) |             0.5210 |
+-------------------+--------------------+


#### Query 2: Probability of Performance (Pass/Fail) given StudyHours = High

In [7]:
query2_result = infer.query(variables=['Performance'], evidence={'StudyHours': 'High'})
print(query2_result)

+-------------------+--------------------+
| Performance       |   phi(Performance) |
+===================+====================+
| Performance(Pass) |             0.7100 |
+-------------------+--------------------+
| Performance(Fail) |             0.2900 |
+-------------------+--------------------+


#### Query 3: Probability of Performance (Pass/Fail) given Attendance = Good and Stress = Low

In [8]:
query3_result = infer.query(variables=['Performance'], evidence={'Attendance': 'Good', 'Stress': 'Low'})
print(query3_result)

+-------------------+--------------------+
| Performance       |   phi(Performance) |
+===================+====================+
| Performance(Pass) |             0.6600 |
+-------------------+--------------------+
| Performance(Fail) |             0.3400 |
+-------------------+--------------------+


#### Query 4: Probability of Stress (High/Low) given Performance = Fail

In [9]:
query4_result = infer.query(variables=['Stress'], evidence={'Performance': 'Fail'})
print(query4_result)

+--------------+---------------+
| Stress       |   phi(Stress) |
+==============+===============+
| Stress(High) |        0.5873 |
+--------------+---------------+
| Stress(Low)  |        0.4127 |
+--------------+---------------+


#### Query 5: Probability of Performance (Fail) given StudyHours = Low, Attendance = Poor, Sleep = Less

In [10]:
query5_result = infer.query(variables=['Performance'], evidence={'StudyHours': 'Low', 'Attendance': 'Poor', 'Sleep': 'Less'})
print(query5_result)

+-------------------+--------------------+
| Performance       |   phi(Performance) |
+===================+====================+
| Performance(Pass) |             0.1200 |
+-------------------+--------------------+
| Performance(Fail) |             0.8800 |
+-------------------+--------------------+
